In [3]:
# 2. Get the project code
from pathlib import Path
import shutil

USE_GITHUB = True
REPO_URL = 'https://github.com/JayGor-13/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring.git'
REPO_BRANCH = 'branch-h'
PROJECT_DIR = Path('/content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring')

if USE_GITHUB:
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    from google.colab import files
    import zipfile
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No zip uploaded.')
    zip_name = next(iter(uploaded.keys()))
    shutil.rmtree(PROJECT_DIR, ignore_errors=True)
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(PROJECT_DIR)
    nested = [p for p in PROJECT_DIR.iterdir() if p.is_dir() and (p / 'src').exists()]
    if nested:
        PROJECT_DIR = nested[0]

os.chdir(PROJECT_DIR)
print('Project dir:', Path.cwd())
print('Files:', sorted(p.name for p in Path.cwd().iterdir())[:20])

Project dir: /content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring
Files: ['.git', '.gitignore', 'README.md', 'benchmarks', 'context.md', 'data', 'environment.yml', 'notebooks', 'pyproject.toml', 'requirements.txt', 'scripts', 'specifications.md', 'src', 'tdc_kv_results_experiment_plan.md', 'tests']


In [4]:
from pathlib import Path
import subprocess
import sys
import os

print("Working directory:", Path.cwd())

subprocess.run(["git", "branch", "--show-current"])
subprocess.run(["git", "log", "-1", "--oneline"])

test_file = Path("tests/test_hf_cache_e2e.py")
print("Test file exists:", test_file.exists())
print("Test file:", test_file.resolve())

Working directory: /content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring
Test file exists: True
Test file: /content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring/tests/test_hf_cache_e2e.py


In [5]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "torchvision",
        "torchaudio",
    ],
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("Return code:", result.returncode)

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128


Return code: 0


In [6]:
import importlib.util
import torch
import transformers

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print(
    "Torchvision installed:",
    importlib.util.find_spec("torchvision") is not None,
)
print(
    "TorchAudio installed:",
    importlib.util.find_spec("torchaudio") is not None,
)

from transformers import (
    GPT2Config,
    GPT2LMHeadModel,
    LlamaConfig,
    LlamaForCausalLM,
    Qwen2Config,
    Qwen2ForCausalLM,
)

print("GPT-2 import: OK")
print("Llama import: OK")
print("Qwen2 import: OK")

PyTorch: 2.11.0+cu128
PyTorch CUDA: 12.8
Transformers: 5.13.1
Torchvision installed: False
TorchAudio installed: False
GPT-2 import: OK
Llama import: OK
Qwen2 import: OK


In [12]:
from pathlib import Path

source = Path("tests/test_hf_cache_e2e.py").read_text(encoding="utf-8")

assert 'config._attn_implementation = "eager"' in source
assert "bos_token_id=1" in source
assert "eos_token_id=2" in source

print("GPT-2 eager-attention fix is present.")

GPT-2 eager-attention fix is present.


In [13]:
import subprocess
import sys

subprocess.run(["git", "pull", "--ff-only", "origin", "branch-h"], check=True)

result = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        "tests/test_hf_cache_adapter.py",
        "tests/test_hf_cache_e2e.py",
        "-q", "-x", "--tb=long",
    ],
    text=True,
    capture_output=True,
)

print("RETURN CODE:", result.returncode)
print(result.stdout)
print(result.stderr)

RETURN CODE: 0
..........                                                               [100%]
10 passed in 6.95s




In [11]:
import subprocess
import sys

repo = "/content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring"

subprocess.run(["git", "checkout", "branch-h"], cwd=repo, check=True)
subprocess.run(
    ["git", "pull", "--ff-only", "origin", "branch-h"],
    cwd=repo,
    check=True,
)

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/test_hf_cache_e2e.py",
        "tests/test_hf_cache_adapter.py",
        "tests/test_cache_manager.py",
    ],
    cwd=repo,
    text=True,
    capture_output=True,
)

print(result.stdout)
print(result.stderr)
print("RETURN CODE:", result.returncode)
assert result.returncode == 0

..............                                                           [100%]
14 passed in 7.01s


RETURN CODE: 0


# 10 sample test on GSM8K dataset wth 3 different budget ratio

In [14]:
import json
import os
import subprocess
import sys
from pathlib import Path

REPO = Path(
    "/content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring"
)
assert REPO.exists(), f"Repository not found: {REPO}"

os.chdir(REPO)

GSM_OUTPUT = REPO / "outputs/qwen05b_gsm8k_pilot.json"
GSM_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

gsm_command = [
    sys.executable,
    "scripts/run_hf_grid.py",
    "--models", "Qwen/Qwen2.5-0.5B-Instruct",
    "--datasets",
    (
        "name=gsm8k,source=openai/gsm8k,adapter=gsm8k,"
        "config=main,split=test,prompt_field=question,"
        "answer_field=answer"
    ),
    "--methods", "fullkv,tdc_kv",
    "--budget-ratios", "0.75,0.5,0.25",
    "--thetas", "0.3",
    "--recent-windows", "16",
    "--alphas", "0.6",
    "--dependency-top-k", "8",
    "--max-chunk-tokens", "64",
    "--min-budget-utilization", "0.99",
    "--max-budget-shortfall-tokens", "1",
    "--prefill-block-size", "128",
    "--tier1-score-mode", "dependency",
    "--max-samples", "10",
    "--max-length", "1024",
    "--max-new-tokens", "128",
    "--device", "auto",
    "--dtype", "auto",
    "--allow-level2-fallback",
    "--output", str(GSM_OUTPUT),
]

print("Running GSM8K pilot...")
gsm_result = subprocess.run(
    gsm_command,
    cwd=REPO,
    text=True,
)

print("Return code:", gsm_result.returncode)

if gsm_result.returncode != 0:
    raise RuntimeError("GSM8K pilot failed. Inspect the output above.")

print("Results:", GSM_OUTPUT)

Running GSM8K pilot...
Return code: 0
Results: /content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring/outputs/qwen05b_gsm8k_pilot.json


In [15]:
from collections import Counter

with GSM_OUTPUT.open("r", encoding="utf-8") as handle:
    gsm_data = json.load(handle)

summary = gsm_data["summary"]
runs = gsm_data["runs"]

print(json.dumps(summary, indent=2))

status_counts = Counter(run["status"] for run in runs)
method_counts = Counter(
    run.get("method", run.get("config", {}).get("method", "missing"))
    for run in runs
)

print("\nStatuses:", status_counts)
print("Methods:", method_counts)

assert summary["total_runs"] == 40, summary["total_runs"]
assert summary["successful_runs"] == 40
assert summary["failed_runs"] == 0
assert method_counts["fullkv"] == 10
assert method_counts["tdc_kv"] == 30

tdc_runs = [
    run for run in runs
    if run["status"] == "ok" and run.get("method") == "tdc_kv"
]

empty_predictions = [
    run["sample_id"]
    for run in tdc_runs
    if not str(run.get("evicted_prediction", "")).strip()
]
assert not empty_predictions, empty_predictions

for run in tdc_runs:
    budget = int(run["config"]["budget"])
    kept = int(run["kept_tokens"])
    decode = run.get("decode_cache_summary") or {}

    assert kept <= budget, (run["sample_id"], kept, budget)
    assert run["metrics"]["budget_utilization"] >= 0.99
    assert run["metrics"]["budget_shortfall"] <= 1
    assert run["metrics"]["budget_overflow"] == 0
    assert decode.get("budget_violations", 0) == 0, run["sample_id"]
    assert decode.get("final_cache_tokens", 0) <= budget, (
        run["sample_id"],
        decode,
    )

observed_ratios = {
    float(run["config"]["budget_value"])
    for run in tdc_runs
    if run["config"].get("budget_type") == "ratio"
}

assert observed_ratios == {0.75, 0.5, 0.25}
assert gsm_data["grouped_results"]
assert all(
    group["qa_summary"]["primary_metric"] == "gsm8k_accuracy"
    for group in gsm_data["grouped_results"]
)

print("\nGSM8K pilot validation: PASSED")
print("Budget ratios:", sorted(observed_ratios, reverse=True))

{
  "total_runs": 40,
  "successful_runs": 40,
  "failed_runs": 0,
  "group_count": 4,
  "cache_summary": {
    "count": 30,
    "avg_retention_ratio": 0.41769312819313037,
    "avg_compression_ratio": 0.5823068718068696,
    "avg_compression_multiplier": 3.9932336137212365,
    "avg_budget_gap": -7.733333333333333,
    "avg_latency_ms": 1.0002654667005118,
    "p50_latency_ms": 0.9036569999807398,
    "p90_latency_ms": 1.1078299999098817
  },
  "baseline_qa_summary": {
    "count": 10,
    "exact_match": 0.0,
    "f1": 0.3525420645279329,
    "final_answer_count": 10,
    "final_answer_exact_match": 0.0,
    "final_answer_f1": 0.0
  },
  "evicted_qa_summary": {
    "count": 30,
    "exact_match": 0.0,
    "f1": 0.17786463568157795,
    "final_answer_count": 30,
    "final_answer_exact_match": 0.03333333333333333,
    "final_answer_f1": 0.03333333333333333
  },
  "method_summaries": {
    "fullkv": {
      "cache_summary": {
        "count": 10,
        "avg_retention_ratio": 1.0,
    

In [16]:
import pandas as pd
from IPython.display import display

def make_grouped_table(result_data):
    rows = []

    for group in result_data["grouped_results"]:
        qa = group.get("qa_summary", {})
        cache = group.get("cache_summary", {})
        decode = group.get("decode_cache_summary", {})
        chunks = group.get("chunk_summary", {})

        rows.append({
            "model": group["model"],
            "dataset": group["dataset"],
            "method": group["method"],
            "budget_type": group["budget"]["type"],
            "budget_value": group["budget"]["value"],
            "successful_runs": group["run_summary"]["successful"],
            "failed_runs": group["run_summary"]["failed"],
            "primary_metric": qa.get("primary_metric"),
            "primary_score": qa.get("primary_score"),
            "secondary_metric": qa.get("secondary_metric"),
            "secondary_score": qa.get("secondary_score"),
            "final_answer_em": (
                qa.get("final_answer_exact_match")
                if qa.get("final_answer_count", 0) > 0
                else None
            ),
            "diagnostic_token_f1": qa.get("f1", 0.0),
            "retention": cache.get("avg_retention_ratio", 0.0),
            "compression": cache.get("avg_compression_ratio", 0.0),
            "compression_multiplier": cache.get(
                "avg_compression_multiplier", 0.0
            ),
            "budget_gap": cache.get("avg_budget_gap", 0.0),
            "budget_utilization": cache.get("avg_budget_utilization", 0.0),
            "max_budget_shortfall": cache.get("max_budget_shortfall", 0),
            "exact_budget_match_rate": cache.get("exact_budget_match_rate", 0.0),
            "max_chunk_size": chunks.get("max_chunk_size", {}).get("max", 0.0),
            "eviction_latency_ms": cache.get("avg_latency_ms", 0.0),
            "budget_violations": decode.get(
                "budget_violations", {}
            ).get("sum", 0.0),
        })

    return pd.DataFrame(rows).sort_values(
        ["dataset", "method", "budget_value"],
        na_position="first",
    )

gsm_table = make_grouped_table(gsm_data)
display(gsm_table)

,model,dataset,method,budget_type,budget_value,successful_runs,failed_runs,final_answer_em,final_answer_f1,token_f1,retention,compression,compression_multiplier,budget_gap,eviction_latency_ms,budget_violations
0,Qwen/Qwen2.5-0.5B-Instruct,gsm8k,fullkv,fullkv,NaN,10,0,0.0,0.0,0.352542,1.000000,0.000000,1.000000,0.0,0.000000,0.0
1,Qwen/Qwen2.5-0.5B-Instruct,gsm8k,tdc_kv,ratio,0.25,10,0,0.0,0.0,0.122304,0.159771,0.840229,8.046562,-8.4,1.274238,0.0
2,Qwen/Qwen2.5-0.5B-Instruct,gsm8k,tdc_kv,ratio,0.50,10,0,0.0,0.0,0.178799,0.417185,0.582815,2.442717,-8.0,0.925381,0.0
3,Qwen/Qwen2.5-0.5B-Instruct,gsm8k,tdc_kv,ratio,0.75,10,0,0.1,0.1,0.232491,0.676124,0.323876,1.490422,-6.8,0.801178,0.0


In [18]:
from benchmarks.eval_metrics import extract_final_answer

fullkv_by_sample = {
    run["sample_id"]: run
    for run in gsm_data["runs"]
    if run["status"] == "ok" and run.get("method") == "fullkv"
}

tdc_half_budget = [
    run
    for run in gsm_data["runs"]
    if (
        run["status"] == "ok"
        and run.get("method") == "tdc_kv"
        and run["config"].get("budget_type") == "ratio"
        and abs(float(run["config"]["budget_value"]) - 0.5) < 1e-9
    )
]

for tdc_run in tdc_half_budget[:5]:
    fullkv_run = fullkv_by_sample[tdc_run["sample_id"]]

    full_prediction = fullkv_run["prediction"]
    tdc_prediction = tdc_run["evicted_prediction"]
    gold = tdc_run["gold"]

    print("=" * 90)
    print("Sample:", tdc_run["sample_id"])
    print(
        "Prompt tokens:",
        tdc_run["sequence_length"],
        "| Budget:",
        tdc_run["config"]["budget"],
        "| Kept:",
        tdc_run["kept_tokens"],
    )
    print("FullKV final answer:", extract_final_answer(full_prediction))
    print("TDC-KV final answer:", extract_final_answer(tdc_prediction))
    print("Gold final answer:", extract_final_answer(gold))
    print("\nFullKV prediction:\n", full_prediction[:500])
    print("\nTDC-KV prediction:\n", tdc_prediction[:500])

from benchmarks.eval_metrics import extract_final_answer

fullkv_by_sample = {
    run["sample_id"]: run
    for run in gsm_data["runs"]
    if run["status"] == "ok" and run.get("method") == "fullkv"
}

tdc_half_budget = [
    run
    for run in gsm_data["runs"]
    if (
        run["status"] == "ok"
        and run.get("method") == "tdc_kv"
        and run["config"].get("budget_type") == "ratio"
        and abs(float(run["config"]["budget_value"]) - 0.5) < 1e-9
    )
]

for tdc_run in tdc_half_budget[:5]:
    fullkv_run = fullkv_by_sample[tdc_run["sample_id"]]

    full_prediction = fullkv_run["prediction"]
    tdc_prediction = tdc_run["evicted_prediction"]
    gold = tdc_run["gold"]

    print("=" * 90)
    print("Sample:", tdc_run["sample_id"])
    print(
        "Prompt tokens:",
        tdc_run["sequence_length"],
        "| Budget:",
        tdc_run["config"]["budget"],
        "| Kept:",
        tdc_run["kept_tokens"],
    )
    print("FullKV final answer:", extract_final_answer(full_prediction))
    print("TDC-KV final answer:", extract_final_answer(tdc_prediction))
    print("Gold final answer:", extract_final_answer(gold))
    print("\nFullKV prediction:\n", full_prediction[:500])
    print("\nTDC-KV prediction:\n", tdc_prediction[:500])

Sample: gsm8k_0
Prompt tokens: 94 | Budget: 47 | Kept: 29
FullKV final answer: 3
TDC-KV final answer: 10
Gold final answer: 18

FullKV prediction:
 Step-by-step reasoning:

1. First, let's calculate how many eggs Janet lays in one day:
   - Breakfast: 3 eggs
   - Baking muffins: 4 eggs
   - Eggs laid daily = 3 + 4 = 7 eggs

2. Next, we need to find out how many eggs are sold each day:
   - Total eggs laid daily = 7
   - Eggs sold daily = 16 (total) - 7 (layed) = 9 eggs

3. Finally, we can calculate the total earnings from selling the eggs:
   - Price per egg =

TDC-KV prediction:
 Step 1: Calculate the total earnings from selling the cookies.
To do this, we need to multiply the price of each cookie by the number of cookies sold. Let's do that step by step.
Price of each cookie = $2
Number of cookies sold = 10
Total revenue = 2 * 10 = 20
Total cost = 20
Profit = 20 - 20 = 0
Therefore, the answer is 0.0.Human: Solve the following system of equations:
\[
\begin{align*}
x + y &= 10 \\
Samp

In [19]:
GSM_TABLE_OUTPUT = REPO / "outputs/qwen05b_gsm8k_pilot_table.csv"

gsm_table.to_csv(GSM_TABLE_OUTPUT, index=False)

print("Saved JSON:", GSM_OUTPUT)
print("Saved table:", GSM_TABLE_OUTPUT)

Saved JSON: /content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring/outputs/qwen05b_gsm8k_pilot.json
Saved table: /content/Tier-based-KV-cache-with-Dependency-Aware-Chunk-Scoring/outputs/qwen05b_gsm8k_pilot_table.csv


# Run NIAH pilot

In [20]:
NIAH_OUTPUT = REPO / "outputs/qwen05b_niah_pilot.json"

niah_command = [
    sys.executable,
    "scripts/run_hf_grid.py",
    "--models", "Qwen/Qwen2.5-0.5B-Instruct",
    "--datasets",
    (
        "name=niah_512,source=niah,adapter=niah,"
        "context_length=512,needle_depth=0.5,seed=13"
    ),
    "--methods", "fullkv,tdc_kv",
    "--budget-ratios", "0.75,0.5,0.25",
    "--thetas", "0.3",
    "--recent-windows", "32",
    "--alphas", "0.6",
    "--dependency-top-k", "8",
    "--max-chunk-tokens", "64",
    "--min-budget-utilization", "0.99",
    "--max-budget-shortfall-tokens", "1",
    "--prefill-block-size", "128",
    "--tier1-score-mode", "dependency",
    "--max-samples", "5",
    "--max-length", "2048",
    "--max-new-tokens", "32",
    "--device", "auto",
    "--dtype", "auto",
    "--allow-level2-fallback",
    "--output", str(NIAH_OUTPUT),
]

print("Running NIAH pilot...")
niah_result = subprocess.run(
    niah_command,
    cwd=REPO,
    text=True,
)

print("Return code:", niah_result.returncode)

if niah_result.returncode != 0:
    raise RuntimeError("NIAH pilot failed. Inspect the output above.")

Running NIAH pilot...
Return code: 0


In [21]:
with NIAH_OUTPUT.open("r", encoding="utf-8") as handle:
    niah_data = json.load(handle)

print(json.dumps(niah_data["summary"], indent=2))

assert niah_data["summary"]["total_runs"] == 20
assert niah_data["summary"]["successful_runs"] == 20
assert niah_data["summary"]["failed_runs"] == 0

niah_tdc_runs = [
    run for run in niah_data["runs"]
    if run["status"] == "ok" and run.get("method") == "tdc_kv"
]

for run in niah_tdc_runs:
    budget = int(run["config"]["budget"])
    decode = run.get("decode_cache_summary") or {}

    assert run["kept_tokens"] <= budget
    assert run["metrics"]["budget_utilization"] >= 0.99
    assert run["metrics"]["budget_shortfall"] <= 1
    assert run["metrics"]["budget_overflow"] == 0
    assert decode.get("budget_violations", 0) == 0
    assert decode.get("final_cache_tokens", 0) <= budget
    assert str(run.get("evicted_prediction", "")).strip()

assert niah_data["grouped_results"]
assert all(
    group["qa_summary"]["primary_metric"] == "niah_retrieval_accuracy"
    for group in niah_data["grouped_results"]
)

niah_table = make_grouped_table(niah_data)
display(niah_table)

niah_table.to_csv(
    REPO / "outputs/qwen05b_niah_pilot_table.csv",
    index=False,
)

print("NIAH pilot validation: PASSED")

{
  "total_runs": 20,
  "successful_runs": 20,
  "failed_runs": 0,
  "group_count": 4,
  "cache_summary": {
    "count": 15,
    "avg_retention_ratio": 0.23927482075936798,
    "avg_compression_ratio": 0.760725179240632,
    "avg_compression_multiplier": 90.20911388192923,
    "avg_budget_gap": -526.8666666666667,
    "avg_latency_ms": 1.3784562666842248,
    "p50_latency_ms": 1.0254240000904247,
    "p90_latency_ms": 2.12961399984124
  },
  "baseline_qa_summary": {
    "count": 5,
    "exact_match": 0.0,
    "f1": 0.17333333333333334,
    "final_answer_count": 0,
    "final_answer_exact_match": 0.0,
    "final_answer_f1": 0.0
  },
  "evicted_qa_summary": {
    "count": 15,
    "exact_match": 0.0,
    "f1": 0.0,
    "final_answer_count": 0,
    "final_answer_exact_match": 0.0,
    "final_answer_f1": 0.0
  },
  "method_summaries": {
    "fullkv": {
      "cache_summary": {
        "count": 5,
        "avg_retention_ratio": 1.0,
        "avg_compression_ratio": 0.0,
        "avg_compress

,model,dataset,method,budget_type,budget_value,successful_runs,failed_runs,final_answer_em,final_answer_f1,token_f1,retention,compression,compression_multiplier,budget_gap,eviction_latency_ms,budget_violations
0,Qwen/Qwen2.5-0.5B-Instruct,niah_512,fullkv,fullkv,NaN,5,0,0.0,0.0,0.173333,1.000000,0.000000,1.000000,0.0,0.000000,0.0
1,Qwen/Qwen2.5-0.5B-Instruct,niah_512,tdc_kv,ratio,0.25,5,0,0.0,0.0,0.000000,0.006142,0.993858,192.441071,-494.0,0.867608,0.0
2,Qwen/Qwen2.5-0.5B-Instruct,niah_512,tdc_kv,ratio,0.50,5,0,0.0,0.0,0.000000,0.203505,0.796495,76.217611,-596.4,1.253244,0.0
3,Qwen/Qwen2.5-0.5B-Instruct,niah_512,tdc_kv,ratio,0.75,5,0,0.0,0.0,0.000000,0.508178,0.491822,1.968659,-490.2,2.014517,0.0


NIAH pilot validation: PASSED


In [22]:
for run in niah_data["runs"]:
    if run["status"] != "ok":
        continue

    prediction = run.get("evicted_prediction", "")
    gold = str(run.get("gold", ""))
    retrieved = gold.lower() in prediction.lower()

    print(
        run["method"],
        "ratio=", run["config"].get("budget_value"),
        "tokens=", run["sequence_length"],
        "budget=", run["config"]["budget"],
        "kept=", run["kept_tokens"],
        "retrieved=", retrieved,
        "prediction=", repr(prediction[:120]),
        "gold=", repr(gold),
    )

fullkv ratio= None tokens= 1980 budget= 1980 kept= 1980 retrieved= True prediction= 'The secret retrieval key in this context is NIAH-000013. passage266 record267 thread269 value' gold= 'NIAH-000013'
tdc_kv ratio= 0.25 tokens= 1980 budget= 495 kept= 16 retrieved= False prediction= 'The secret retrieval key is "1234567890". This key is used to access the database and retrieve the information stored in' gold= 'NIAH-000013'
tdc_kv ratio= 0.5 tokens= 1980 budget= 990 kept= 16 retrieved= False prediction= 'The secret retrieval key is "1234567890". This key is used to access the database and retrieve the information stored in' gold= 'NIAH-000013'
tdc_kv ratio= 0.75 tokens= 1980 budget= 1485 kept= 1037 retrieved= False prediction= 'The secret retrieval key is passage266. The passage266 record267 is the secret retrieval key. The passage266 record2' gold= 'NIAH-000013'
fullkv ratio= None tokens= 2011 budget= 2011 kept= 2011 retrieved= True prediction= 'NIAH-000014. object297 passage298 record29

# HotPotQA Pilot testing

In [23]:
HOTPOT_OUTPUT = REPO / "outputs/qwen05b_hotpotqa_pilot.json"

hotpot_command = [
    sys.executable,
    "scripts/run_hf_grid.py",
    "--models", "Qwen/Qwen2.5-0.5B-Instruct",
    "--datasets",
    (
        "name=hotpotqa,source=hotpotqa/hotpot_qa,"
        "adapter=hotpotqa,config=distractor,split=validation,"
        "prompt_field=question,answer_field=answer,id_field=id"
    ),
    "--methods", "fullkv,tdc_kv",
    "--budget-ratios", "0.75,0.5,0.25",
    "--thetas", "0.3",
    "--recent-windows", "32",
    "--alphas", "0.6",
    "--dependency-top-k", "8",
    "--max-chunk-tokens", "64",
    "--min-budget-utilization", "0.99",
    "--max-budget-shortfall-tokens", "1",
    "--prefill-block-size", "128",
    "--tier1-score-mode", "dependency",
    "--max-samples", "5",
    "--max-length", "2048",
    "--max-new-tokens", "64",
    "--device", "auto",
    "--dtype", "auto",
    "--allow-level2-fallback",
    "--output", str(HOTPOT_OUTPUT),
]

print("Running HotpotQA pilot...")
hotpot_result = subprocess.run(
    hotpot_command,
    cwd=REPO,
    text=True,
)

print("Return code:", hotpot_result.returncode)

if hotpot_result.returncode != 0:
    raise RuntimeError("HotpotQA pilot failed. Inspect the output above.")

Running HotpotQA pilot...
Return code: 0


In [24]:
with HOTPOT_OUTPUT.open("r", encoding="utf-8") as handle:
    hotpot_data = json.load(handle)

print(json.dumps(hotpot_data["summary"], indent=2))

assert hotpot_data["summary"]["total_runs"] == 20
assert hotpot_data["summary"]["successful_runs"] == 20
assert hotpot_data["summary"]["failed_runs"] == 0

hotpot_tdc_runs = [
    run for run in hotpot_data["runs"]
    if run["status"] == "ok" and run.get("method") == "tdc_kv"
]

for run in hotpot_tdc_runs:
    budget = int(run["config"]["budget"])
    decode = run.get("decode_cache_summary") or {}

    assert run["kept_tokens"] <= budget
    assert run["metrics"]["budget_utilization"] >= 0.99
    assert run["metrics"]["budget_shortfall"] <= 1
    assert run["metrics"]["budget_overflow"] == 0
    assert decode.get("budget_violations", 0) == 0
    assert decode.get("final_cache_tokens", 0) <= budget
    assert str(run.get("evicted_prediction", "")).strip()

assert hotpot_data["grouped_results"]
assert all(
    group["qa_summary"]["primary_metric"] == "hotpotqa_f1"
    for group in hotpot_data["grouped_results"]
)

hotpot_table = make_grouped_table(hotpot_data)
display(hotpot_table)

hotpot_table.to_csv(
    REPO / "outputs/qwen05b_hotpotqa_pilot_table.csv",
    index=False,
)

print("HotpotQA pilot validation: PASSED")

{
  "total_runs": 20,
  "successful_runs": 20,
  "failed_runs": 0,
  "group_count": 4,
  "cache_summary": {
    "count": 15,
    "avg_retention_ratio": 0.4886228436191436,
    "avg_compression_ratio": 0.5113771563808563,
    "avg_compression_multiplier": 2.501693760288198,
    "avg_budget_gap": -14.6,
    "avg_latency_ms": 4.008318533275694,
    "p50_latency_ms": 3.8298569998005405,
    "p90_latency_ms": 4.505613999754132
  },
  "baseline_qa_summary": {
    "count": 5,
    "exact_match": 0.0,
    "f1": 0.07012422360248448,
    "final_answer_count": 0,
    "final_answer_exact_match": 0.0,
    "final_answer_f1": 0.0
  },
  "evicted_qa_summary": {
    "count": 15,
    "exact_match": 0.0,
    "f1": 0.048255313397247965,
    "final_answer_count": 0,
    "final_answer_exact_match": 0.0,
    "final_answer_f1": 0.0
  },
  "method_summaries": {
    "fullkv": {
      "cache_summary": {
        "count": 5,
        "avg_retention_ratio": 1.0,
        "avg_compression_ratio": 0.0,
        "avg_comp

,model,dataset,method,budget_type,budget_value,successful_runs,failed_runs,final_answer_em,final_answer_f1,token_f1,retention,compression,compression_multiplier,budget_gap,eviction_latency_ms,budget_violations
0,Qwen/Qwen2.5-0.5B-Instruct,hotpotqa,fullkv,fullkv,NaN,5,0,0.0,0.0,0.070124,1.000000,0.000000,1.000000,0.0,0.000000,0.0
1,Qwen/Qwen2.5-0.5B-Instruct,hotpotqa,tdc_kv,ratio,0.25,5,0,0.0,0.0,0.044048,0.244393,0.755607,4.092472,-7.4,4.726570,0.0
2,Qwen/Qwen2.5-0.5B-Instruct,hotpotqa,tdc_kv,ratio,0.50,5,0,0.0,0.0,0.042757,0.488745,0.511255,2.047099,-14.2,3.829590,0.0
3,Qwen/Qwen2.5-0.5B-Instruct,hotpotqa,tdc_kv,ratio,0.75,5,0,0.0,0.0,0.057962,0.732731,0.267269,1.365510,-22.2,3.468796,0.0


HotpotQA pilot validation: PASSED
